# Guessing the most unique country name by English Language

So this is inspired by [
Eliminating Country Names Until I Find The Most Unique](https://www.youtube.com/watch?v=tnwMre_cVkc) YouTube Video by [
Name Explain](https://www.youtube.com/@NameExplain) uploaded on 12 Dec 2025. Towards the end of the video in final elimination round the Youtuber uses mean by frequency of letters in English Language as described in [Letter Frequencies in English](https://mathcenter.oxford.emory.edu/site/math125/englishLetterFreqs/) published as part of Math125 Course offered by Oxford College's Department of Mathematics and Computer Science.<sup>[\[Archived Link\]](https://web.archive.org/web/20260409183439/https://mathcenter.oxford.emory.edu/site/math125/englishLetterFreqs/)</sup>. The list of countries is same as [UN Official Name of Countries](https://www.un.int/protocol/sites/www.un.int/files/Protocol%20and%20Liaison%20Service/officialnamesofcountries.pdf).<sup>[\[Archived Link\]](https://web.archive.org/web/20260429225736/https://www.un.int/protocol/sites/www.un.int/files/Protocol%20and%20Liaison%20Service/officialnamesofcountries.pdf)</sup>.

Now there is already a measure of how rare a sequence of letters (or words or any sequence) is, it is called [perplexity](https://d2l.ai/chapter_recurrent-neural-networks/language-model.html#perplexity). Roughly it is measure of how likely is one to get a random piece of letters, the amount of surprise it produces *i.e. how unique it is*. Higher it is, the more unique it is.  

$\frac{1}{n} \sum_{t=1}^n -\log P(x_t \mid x_{t-1}, \ldots, x_1)$

Now we can only predict the frequency of one letter from the given distribution, so assume zero order markov model and it becomes

$\frac{1}{n} \sum_{t=1}^n -\log P(x_t)$

## Steps

### 1. Download the list of countries
### 2. Extract the table of frequency
### 3. Calculate perplexity of each country
### 4. Visualize the histogram

### 1. Download the list of countries

In [16]:
un_country_list_url = 'https://www.un.int/protocol/sites/www.un.int/files/Protocol%20and%20Liaison%20Service/officialnamesofcountries.pdf'

In [45]:
%%capture
!pip install pypdf
!pip install playwright
!playwright install
!apt-get update && apt-get install -y libatk1.0-0 libatk-bridge2.0-0 libgtk-3-0 libgbm-dev

In [69]:
import pypdf
from playwright.async_api import async_playwright
from io import BytesIO
import re

In [66]:
async with async_playwright() as p:
      browser = await p.chromium.launch()
      page = await browser.new_page()
      resp = await page.request.get(un_country_list_url)
      raw_pdf_bytes = await resp.body()
      await browser.close()

In [68]:
pdf_io = BytesIO(raw_pdf_bytes)
pdf_io.seek(0)
pdf_reader = pypdf.PdfReader(pdf_io)

In [71]:
total_pdf_content = ''

In [72]:
for page in pdf_reader.pages:
    total_pdf_content += page.extract_text()


In [74]:
total_pdf_content[:100]

'OFFICIAL NAMES OF THE UNITED NATIONS MEMBERSHIP \n \nIslamic Republic of Afghanistan  \nRepublic of Alb'

In [92]:
list_of_countries = total_pdf_content.split('\n')

In [93]:
list_of_countries = [re.sub(r'\d', '', country) for country in list_of_countries]

In [94]:
list_of_countries = [list_of_countries[i].strip() for i in range(1, len(list_of_countries)) if list_of_countries[i].strip() != '']

In [96]:
list_of_countries[:10]

['Islamic Republic of Afghanistan',
 'Republic of Albania',
 'People’s Democratic Republic of Algeria',
 'Principality of Andorra',
 'Republic of Angola',
 'Antigua and Barbuda',
 'Republic of Argentina',
 'Republic of Armenia',
 'Commonwealth of  Australia',
 'Republic of Austria']

## 2. Extract Table of Frequency

In [82]:
%%capture
!pip install bs4

In [85]:
from bs4 import BeautifulSoup
import requests

In [84]:
freq_url = 'https://www.google.com/url?q=https%3A%2F%2Fmathcenter.oxford.emory.edu%2Fsite%2Fmath125%2FenglishLetterFreqs%2F'

In [86]:
bs_parser = BeautifulSoup(requests.get(freq_url).text, 'html.parser')

In [87]:
bs_parser

<html lang="en"><head><meta content="text/html; charset=utf-8" http-equiv="Content-Type"/><title>Redirect Notice</title><style>body,div,a{font-family:sans-serif}body{background-color:#fff;margin-top:3px}div{color:#1f1f1f}a:link{color:#681da8}a:visited{color:#681da8}a:active{color:#ea4335}div.mymGo{border-top:1px solid #d2d2d2;border-bottom:1px solid #d2d2d2;background:#f3f5f6;margin-top:1em;width:100%}div.aXgaGb{padding:0.5em 0;margin-left:10px}div.fTk7vd{margin-left:35px;margin-top:35px}</style></head><body><div class="mymGo"><div class="aXgaGb"><font style="font-size:larger"><b>Redirect Notice</b></font></div></div><div class="fTk7vd"> The previous page is sending you to <a href="https://mathcenter.oxford.emory.edu/site/math125/englishLetterFreqs/">https://mathcenter.oxford.emory.edu/site/math125/englishLetterFreqs/</a>.<br/><br/> If you do not want to visit that page, you can <a href="#" id="tsuid_c_oBar7pLveSmLQPz7bc4Ak_1">return to the previous page</a>.<script nonce="VUY2SKFW7uJ4